# All-atom DPD + FastFIRE initialization

This notebook is the all-atom adaptation of the earlier coarse-grained DPD examples: build a small polymer melt, relax overlaps with a physical-unit AA-DPD initializer, hand the coordinates back to the original OpenFF model, and run a short stability check.

What this demonstrates:

- OpenFF 2.3.0 (Sage) provides the bonded terms, nonbonded atom types, true atomic masses, and final OpenMM force field.
- HOOMD initialization uses physical Å length, amu mass, and kcal/mol energy conventions, not reduced bead units.
- The DPD `kT`, `dt`, and `gamma` values are empirical initialization controls; the DPD trajectory is not physical molecular dynamics.
- Bond, angle, torsion, and improper coefficients are multiplied by **30× only during initialization** to preserve local geometry while overlaps are removed.
- DPD repulsions use atom-type epsilon scaling from the OpenFF vdW labels.
- The initializer runs stochastic DPD first, then conservative FIRE minimization.
- After initialization, OpenMM uses the original unscaled OpenFF model.

The short NVT/NPT molecular dynamics at the end is a **finite-energy handoff stability check, not equilibration**.


## Imports

**One-time setup before running this notebook:** from the repository root, run `conda env create -f environment-gpu.yml`, `conda activate phantomwalk-gpu`, and `python -m pip install --no-deps -e .`. Register it with `python -m ipykernel install --user --name phantomwalk-gpu --display-name "PhantomWalk GPU"`, select that kernel for this notebook, and restart it before running the cells.

The import cell below expects PhantomWalk to be installed editable in the selected environment and intentionally does not use `sys.path` hacks.


In [ ]:
import time
from pathlib import Path
from dataclasses import asdict

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from phantomwalk.benchmarks.aa_test_systems import (
    build_melt,
    minimum_nonbonded_distance_a,
    shortest_periodic_one_four_distance_a,
    write_visualization_pdb,
)
from phantomwalk.lib.all_atom import (
    create_interchange,
    minimize_interchange,
    run_interchange_dynamics,
    update_compound_positions,
)
from phantomwalk.lib.fastfire import AllAtomFastFIRESettings, run_all_atom_fastfire

try:
    import py3Dmol
except ImportError:
    py3Dmol = None

plt.style.use("ggplot")


## User settings

Edit only this cell for routine exploration. PES uses a five-monomer repeat pattern, so `CHAIN_LENGTH` must be divisible by 5 for `CHEMISTRY = "pes"`. The DPD values below are the validated defaults used by this workflow; exposing them here allows controlled hyperparameter sweeps without changing the library implementation.


In [ ]:
# --- USER SETTINGS ---------------------------------------------------------
CHEMISTRY = "p3ht"       # choices: "pe", "p3ht", "pes"
N_CHAINS = 8
CHAIN_LENGTH = 10        # PES must be divisible by 5
DENSITY_G_CM3 = 1.1      # suggested: PE/P3HT 1.1, PES 1.3
SEED = 11
DEVICE = "GPU"           # choices: "GPU", "CPU", "auto"
FULL_PROTOCOL = True
NVT_STEPS = 1000
NPT_STEPS = 1000
OUTPUT_DIR = Path("aa_dpd_showcase")
SHOW_STRUCTURES = True

# Validated AA-DPD defaults; change these only for controlled parameter studies.
DPD_A = 5000.0             # conservative repulsion A (kcal/mol/Å)
DPD_BONDED_SCALE = 30.0    # multiplier applied to OpenFF bonded force constants
DPD_GAMMA = 800.0          # DPD dissipative coefficient
DPD_DT = 0.002             # HOOMD integration timestep
DPD_R_CUT = 3.5            # DPD cutoff (Å)
DPD_KT = 1.0               # DPD thermal energy (kcal/mol)
# ---------------------------------------------------------------------------


## Protocol controls

`FULL_PROTOCOL = True` uses the default all-atom FastFIRE settings. For quick, nonproduction previews, set it to `False`; the preview intentionally runs only 250 DPD steps and at most 300 FIRE steps, with convergence requirements disabled.

The initializer is expressed in physical Å/amu/kcal/mol conventions so OpenFF-derived geometry and masses carry through consistently. The DPD thermostat-like parameters are still empirical packing controls, so interpret the DPD stage as overlap removal rather than a physical trajectory.


In [ ]:
if CHEMISTRY not in {"pe", "p3ht", "pes"}:
    raise ValueError('CHEMISTRY must be one of "pe", "p3ht", or "pes"')
if CHEMISTRY == "pes" and CHAIN_LENGTH % 5:
    raise ValueError("PES CHAIN_LENGTH must be divisible by 5")
if DEVICE not in {"GPU", "CPU", "auto"}:
    raise ValueError('DEVICE must be "GPU", "CPU", or "auto"')
if DEVICE == "GPU":
    import hoomd
    if not hoomd.version.gpu_enabled:
        raise RuntimeError(
            "This kernel has a CPU-only HOOMD build. Select the 'PhantomWalk GPU' kernel and restart."
        )

dpd_parameters = dict(
    repulsion=DPD_A,
    bonded_scale=DPD_BONDED_SCALE,
    gamma=DPD_GAMMA,
    dt=DPD_DT,
    r_cut=DPD_R_CUT,
    kT=DPD_KT,
)

if FULL_PROTOCOL:
    settings = AllAtomFastFIRESettings(seed=SEED, device=DEVICE, **dpd_parameters)
    protocol_label = "full default AA-DPD + FastFIRE protocol"
else:
    settings = AllAtomFastFIRESettings(
        seed=SEED,
        device=DEVICE,
        **dpd_parameters,
        dpd_steps=250,
        dpd_max_steps=250,
        require_dpd_convergence=False,
        fire_steps=100,
        fire_max_steps=300,
        require_fire_convergence=False,
    )
    protocol_label = "nonproduction preview: DPD 250/max250, FIRE 100/max300, convergence disabled"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

display(Markdown(f"**Protocol:** {protocol_label}"))
display(Markdown(f"**Settings:** `{asdict(settings)}`"))


## Build the requested all-atom melt

The builder creates exactly `N_CHAINS` chains of degree `CHAIN_LENGTH`, places them in a cubic periodic box sized from the true atomic masses, and stores coordinates in nanometers in the mBuild compound. The FastFIRE initializer converts to Å internally.


In [ ]:
AMU_NM3_TO_G_CM3 = 1.66053906660e-3

build_start = time.perf_counter()
compound = build_melt(
    CHEMISTRY,
    n_chains=N_CHAINS,
    degree=CHAIN_LENGTH,
    density_g_cm3=DENSITY_G_CM3,
    seed=SEED,
)
construction_s = time.perf_counter() - build_start

particles = list(compound.particles())
masses_amu = np.array([float(p.mass) for p in particles], dtype=float)
box_lengths_nm = np.asarray(compound.box.lengths, dtype=float)
volume_nm3 = float(np.prod(box_lengths_nm))
actual_density = float(masses_amu.sum() * AMU_NM3_TO_G_CM3 / volume_nm3)
finite_initial = bool(np.all(np.isfinite(np.asarray(compound.xyz, dtype=float))))

print(f"Chemistry:        {CHEMISTRY}")
print(f"Chains:           {N_CHAINS}")
print(f"Degree:           {CHAIN_LENGTH}")
print(f"Actual atoms:     {compound.n_particles}")
print(f"Box lengths:      {box_lengths_nm} nm")
print(f"Target density:   {DENSITY_G_CM3:.3f} g/cm^3")
print(f"Actual density:   {actual_density:.3f} g/cm^3")
print(f"Build time:       {construction_s:.2f} s")
print(f"Finite coords:    {finite_initial}")

if not finite_initial:
    raise ValueError("Initial coordinates contain non-finite values")
if float(np.min(box_lengths_nm)) <= 2.0:
    raise ValueError(
        "The box side is <= 2.0 nm. Increase N_CHAINS before creating the OpenMM "
        "system so the nonbonded cutoff has enough periodic headroom."
    )


## Save and optionally preview the initial structure

PDB files are written for visualization only. For large systems, open the saved file in VMD, OVITO, or another molecular viewer instead of rendering inline.


In [ ]:
def display_structure(path: Path, max_atoms: int = 4000) -> None:
    """Render a small PDB inline when py3Dmol is available."""
    if not SHOW_STRUCTURES:
        print(f"Wrote {path}")
        return
    if py3Dmol is None:
        print(f"py3Dmol is not installed; open {path} in an external viewer.")
        return
    if compound.n_particles > max_atoms:
        print(f"System has {compound.n_particles} atoms; open {path} in an external viewer.")
        return
    view = py3Dmol.view(width=700, height=450)
    view.addModel(path.read_text(), "pdb")
    view.setStyle({"stick": {"radius": 0.12}})
    view.addUnitCell()
    view.zoomTo()
    display(view)

initial_pdb = OUTPUT_DIR / "initial.pdb"
write_visualization_pdb(compound, initial_pdb)
display_structure(initial_pdb)


## Create the OpenFF Interchange

Interchange construction is timed separately from the HOOMD AA-DPD/FastFIRE parameterization so the table below can show where time is spent.


In [ ]:
interchange_start = time.perf_counter()
interchange = create_interchange(compound)
interchange_s = time.perf_counter() - interchange_start
print(f"OpenFF/Interchange construction: {interchange_s:.2f} s")


## Run all-atom DPD followed by conservative FIRE

The same Interchange object is passed in so the final FastFIRE coordinates are copied back for the subsequent OpenMM minimization. Single-frame GSD files are optional checkpoints for inspecting the initial, post-DPD, and post-FIRE states.


In [ ]:
fastfire_start = time.perf_counter()
result = run_all_atom_fastfire(
    compound,
    settings,
    interchange=interchange,
    initial_gsd=OUTPUT_DIR / "initial.gsd",
    post_dpd_gsd=OUTPUT_DIR / "post_dpd.gsd",
    post_fire_gsd=OUTPUT_DIR / "post_fire.gsd",
)
fastfire_wall_s = time.perf_counter() - fastfire_start
combined_s = result.dpd_s + result.fire_s

rows = [
    ("mBuild construction", f"{construction_s:.2f} s", "requested melt"),
    ("OpenFF/Interchange", f"{interchange_s:.2f} s", "original OpenFF 2.3.0"),
    ("HOOMD parameterization", f"{result.parameterization_s:.2f} s", "physical Å, true masses"),
    ("HOOMD setup", f"{result.setup_s:.2f} s", result.device_description),
    (
        "DPD",
        f"{result.dpd_steps} steps / {result.dpd_s:.2f} s",
        f"{result.dpd_steps / max(result.dpd_s, 1e-12):.0f} TPS; "
        f"empirical energy-stop criterion satisfied={result.dpd_converged}",
    ),
    ("FIRE", f"{result.fire_steps} steps / {result.fire_s:.2f} s", f"converged={result.fire_converged}"),
    ("DPD + FIRE", f"{combined_s:.2f} s", f"wall around call={fastfire_wall_s:.2f} s"),
]

table = "| Stage | Time / steps | Notes |\n|---|---:|---|\n" + "\n".join(
    f"| {stage} | {value} | {notes} |" for stage, value, notes in rows
)
display(Markdown(table))
display(Markdown(f"**Actual HOOMD device:** `{result.device_description}`"))
print("Bonded interaction counts:", result.bonded_counts)
print("Reference epsilon:", result.epsilon_ref_kcal_mol, "kcal/mol")


## Plot DPD energy convergence history

Total energy grows with system size, so we normalize the DPD pair energy by the number of atoms and each bonded energy by the number of corresponding interactions. The plot calls these quantities *normalized energies*; they have units of kcal/mol but are not equilibrium thermodynamic observables because this initialization uses artificial DPD repulsion and scaled bonded forces.

We define DPD convergence as **every normalized energy component changing by no more than 2% between checks for two consecutive checks**, after the requested minimum number of DPD steps. The first panel exposes the values used by that stopping criterion.

The number of DPD steps is expected to vary between melts. The random-walk placement determines the initial overlap pattern, and chemistry, topology, and stochastic DPD fluctuations also affect how quickly the energy changes settle below the threshold. The 10,000-step maximum is a safety cap rather than a physical relaxation time: reaching it means only that this empirical two-check criterion was not satisfied before the cap, not by itself that the resulting structure is invalid. Inspect the energy history and post-FIRE structure before deciding whether to increase the limit.

For comparison with coarse-grained workflows, the second panel reports each component and their total as the dimensionless quantity $E/(N\epsilon_{\max})$, often described as energy in units of $\epsilon$ per atom. Here $\epsilon_{\max}$ is the largest OpenFF vdW epsilon assigned to any atom type in the system.


In [ ]:
history = result.dpd_energy_history
if not history:
    print("No DPD energy history was recorded.")
else:
    steps = np.array([row["step"] for row in history], dtype=float)
    convergence_keys = [key for key in history[0] if key.endswith("_energy_per_atom") or key.endswith("_energy_per_interaction")]
    fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
    epsilon_per_atom = {}
    for key in convergence_keys:
        # Pair energy is per atom; bonded energies are per interaction.
        values = np.array([row.get(key, np.nan) for row in history], dtype=float)
        axes[0].plot(steps, values, marker="o", label=key.replace("_", " "))
        kind = key.split("_energy_", 1)[0]
        if key.endswith("_per_atom"):
            epsilon_per_atom[kind] = values / result.epsilon_ref_kcal_mol
        else:
            epsilon_per_atom[kind] = (
                values * result.bonded_counts[kind]
                / (result.n_particles * result.epsilon_ref_kcal_mol)
            )
    epsilon_per_atom["total"] = np.sum(list(epsilon_per_atom.values()), axis=0)
    for kind, values in epsilon_per_atom.items():
        line_kwargs = {"color": "black", "linewidth": 2.5} if kind == "total" else {}
        axes[1].plot(steps, values, marker="o", label=kind, **line_kwargs)
    axes[0].set_ylabel("Normalized energy (kcal/mol)")
    axes[0].set_title("Quantities used by the 2% convergence test")
    axes[1].set_xlabel("DPD step")
    axes[1].set_ylabel(r"$E/(N\epsilon_{max})$")
    axes[1].set_title(r"CG-style energy in $\epsilon_{max}$ per atom")
    for ax in axes:
        ax.legend()
    fig.tight_layout()
    plt.show()


## Inspect the post-FIRE structure

After FastFIRE, coordinates in the mBuild compound have been updated in place. We now check the closest nonexcluded periodic distance and write a `post_fire.pdb` directly from that updated compound.


In [ ]:
post_fire_distance_a = minimum_nonbonded_distance_a(compound)
post_fire_one_four_distance_a = shortest_periodic_one_four_distance_a(compound)
post_fire_pdb = OUTPUT_DIR / "post_fire.pdb"
write_visualization_pdb(compound, post_fire_pdb)
print(f"Minimum nonexcluded distance after FIRE: {post_fire_distance_a:.3f} Å")
print(f"Shortest exact 1-4 distance after FIRE:  {post_fire_one_four_distance_a:.3f} Å")
display_structure(post_fire_pdb)


## Minimize with the original unscaled OpenFF model in OpenMM

This is an uncapped local minimization (`max_iterations=0`) on the matching OpenMM platform. `GPU` maps to `CUDA`, `CPU` maps to `CPU`, and `auto` lets the helper choose an available platform.

The ε/atom values below are a rough normalization only: OpenMM total potential energy includes bonded terms, electrostatics, and arbitrary energy offsets, while `epsilon_ref` is just the largest OpenFF vdW epsilon used by the initializer.


In [ ]:
def openmm_platform_from_device(device: str) -> str:
    if device == "GPU":
        return "CUDA"
    if device == "CPU":
        return "CPU"
    return "auto"

openmm_platform = openmm_platform_from_device(DEVICE)
minimization = minimize_interchange(
    interchange,
    max_iterations=0,
    platform_name=openmm_platform,
)

initial_total_kj_mol = minimization.initial_energy_kj_mol
final_total_kj_mol = minimization.minimized_energy_kj_mol
delta_total_kj_mol = final_total_kj_mol - initial_total_kj_mol
epsilon_kj_mol = result.epsilon_ref_kcal_mol * 4.184
initial_epsilon_per_atom = initial_total_kj_mol / (compound.n_particles * epsilon_kj_mol)
final_epsilon_per_atom = final_total_kj_mol / (compound.n_particles * epsilon_kj_mol)
delta_epsilon_per_atom = delta_total_kj_mol / (compound.n_particles * epsilon_kj_mol)

if not minimization.finite or not np.all(np.isfinite([initial_total_kj_mol, final_total_kj_mol])):
    raise RuntimeError("OpenMM minimization produced non-finite energies")
if final_total_kj_mol > initial_total_kj_mol:
    raise RuntimeError(
        "OpenMM minimization increased the total energy: "
        f"initial={initial_total_kj_mol:.6e} kJ/mol, "
        f"final={final_total_kj_mol:.6e} kJ/mol"
    )

update_compound_positions(compound, interchange)
post_min_distance_a = minimum_nonbonded_distance_a(compound)
post_min_pdb = OUTPUT_DIR / "post_minimization.pdb"
write_visualization_pdb(compound, post_min_pdb)

print(f"OpenMM platform:              {minimization.platform_name}")
print(f"Finite energies:              {minimization.finite}")
print(f"Initial total energy:         {initial_total_kj_mol: .6e} kJ/mol")
print(f"Final total energy:           {final_total_kj_mol: .6e} kJ/mol")
print(f"Δ total energy:               {delta_total_kj_mol: .6e} kJ/mol")
print(f"OpenFF ε reference:           {epsilon_kj_mol: .6e} kJ/mol")
print(f"Initial energy / (ε·atom):    {initial_epsilon_per_atom: .6e}")
print(f"Final energy / (ε·atom):      {final_epsilon_per_atom: .6e}")
print(f"Δ energy / (ε·atom):          {delta_epsilon_per_atom: .6e}")
print(f"Minimum nonexcluded distance: {post_min_distance_a:.3f} Å")
print(f"Minimization time:            {minimization.elapsed_s:.2f} s")
display_structure(post_min_pdb)


## Short NVT/NPT stability check

Run a short 300 K, 1 bar, 2 fs OpenMM dynamics check only after minimization. This checks whether the handoff remains finite for a brief trajectory; it is not intended to equilibrate density, conformations, or thermodynamic observables.


In [ ]:
dynamics = run_interchange_dynamics(
    interchange,
    nvt_steps=NVT_STEPS,
    npt_steps=NPT_STEPS,
    temperature_k=300.0,
    pressure_bar=1.0,
    timestep_fs=2.0,
    seed=SEED,
    platform_name=openmm_platform,
)

final_log = dynamics.npt_log[-1] if dynamics.npt_log else (dynamics.nvt_log[-1] if dynamics.nvt_log else {})
final_temperature_k = final_log.get("temperature_k", float("nan"))
final_density_g_cm3 = final_log.get("density_g_cm3", float("nan"))

if not dynamics.finite:
    raise RuntimeError("OpenMM short dynamics produced non-finite log values")
if not np.all(np.isfinite([final_temperature_k, final_density_g_cm3])):
    raise RuntimeError(
        "OpenMM short dynamics ended with non-finite final temperature or density: "
        f"temperature={final_temperature_k}, density={final_density_g_cm3}"
    )

update_compound_positions(compound, interchange)
post_npt_distance_a = minimum_nonbonded_distance_a(compound)
if not np.isfinite(post_npt_distance_a):
    raise RuntimeError("OpenMM short dynamics ended with a non-finite minimum nonexcluded distance")
post_npt_pdb = OUTPUT_DIR / "post_npt.pdb"
write_visualization_pdb(compound, post_npt_pdb)

print(f"OpenMM platform:                  {dynamics.platform_name}")
print(f"Finite log values:                {dynamics.finite}")
print(f"NVT time:                         {dynamics.nvt_s:.2f} s")
print(f"NPT time:                         {dynamics.npt_s:.2f} s")
print(f"Final temperature:                {final_temperature_k:.2f} K")
print(f"Final density:                    {final_density_g_cm3:.3f} g/cm^3")
print(f"Minimum nonexcluded distance:     {post_npt_distance_a:.3f} Å")
print(f"Wrote:                            {post_npt_pdb}")


## Plot the short dynamics logs

The NPT step axis is offset by `NVT_STEPS`, and the two phase logs are joined into one cumulative curve. The vertical line marks the switch from NVT to NPT; joining the logs avoids a visual gap between two separately drawn line objects.


In [ ]:
def series(log, key):
    return np.array([row[key] for row in log], dtype=float)

if not dynamics.nvt_log and not dynamics.npt_log:
    print("No dynamics log records were collected.")
else:
    nvt_steps = series(dynamics.nvt_log, "step") if dynamics.nvt_log else np.array([])
    npt_steps = (series(dynamics.npt_log, "step") + NVT_STEPS) if dynamics.npt_log else np.array([])
    cumulative_steps = np.concatenate((nvt_steps, npt_steps))

    fig, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
    metrics = [
        ("potential_energy_per_atom_kj_mol", "Potential energy / atom (kJ/mol)"),
        ("temperature_k", "Temperature (K)"),
        ("density_g_cm3", "Density (g/cm³)"),
    ]
    for ax, (key, ylabel) in zip(axes, metrics):
        values = np.concatenate((series(dynamics.nvt_log, key), series(dynamics.npt_log, key)))
        ax.plot(cumulative_steps, values, marker="o")
        if dynamics.nvt_log and dynamics.npt_log:
            ax.axvline(NVT_STEPS, color="black", linestyle="--", alpha=0.6, label="NVT → NPT")
        ax.set_ylabel(ylabel)
        if dynamics.nvt_log and dynamics.npt_log:
            ax.legend()
    axes[-1].set_xlabel("Cumulative MD step")
    fig.suptitle("Short OpenMM handoff stability check")
    fig.tight_layout()
    plt.show()


## Summary

If the minimization and short NVT/NPT run report finite energies, coordinates, temperature, and density, the initialized structure is stable enough for handoff to a normal OpenFF/OpenMM workflow. That success does **not** mean the melt is equilibrated.


In [ ]:
expected_files = [
    "initial.pdb",
    "initial.gsd",
    "post_dpd.gsd",
    "post_fire.gsd",
    "post_fire.pdb",
    "post_minimization.pdb",
    "post_npt.pdb",
]

for name in expected_files:
    file_path = OUTPUT_DIR / name
    status = "present" if file_path.exists() else "not written yet"
    print(f"{file_path}: {status}")

finite_coords_after_md = bool(np.all(np.isfinite(np.asarray(compound.xyz, dtype=float))))
print(f"Finite coordinates after short MD: {finite_coords_after_md}")
print("Finite short MD indicates handoff stability only; continue with a real equilibration protocol before production analysis.")
